# Video Game Sales

**Author:** Aditya Krishnan Radhakrishnan  
**Course:** Math 10, UC Irvine, Fall 2022

## Introduction

The goal of this project is to analyze what factors affect the sales of popular video games in the past couple decades. The project also aims to explore regional preferences of games and create a model to predict the release platform of a game. This analysis will incorporate data manipulation using the Pandas library, plotting graphs using Altair, and some machine learning algorithms.

In [ ]:
import pandas as pd
import altair as alt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

## Exploring and Cleaning the Data

We start by loading the dataset from Kaggle ([Video Game Sales by GregorySmith](https://www.kaggle.com/datasets/gregorut/videogamesales)) and dropping rows with missing values.

In [ ]:
df = pd.read_csv("vgsales.csv")
df.dropna(axis=0, inplace=True)
df.info()

In [ ]:
df.head()

There are still some games in the dataset with "Unknown" publishers, so we use boolean indexing to remove those rows.

In [ ]:
df = df[df["Publisher"] != 'Unknown'].copy()

The `Rank` column is redundant given the index, so we drop it.

In [ ]:
df.drop(['Rank'], axis=1, inplace=True)

Convert the `Year` column from float to datetime.

In [ ]:
df['Year'] = df['Year'].map(lambda x: int(x)).copy()
df['Year'] = df['Year'].map(lambda x: f"{x}-01-01").copy()
df['Year'] = pd.to_datetime(df['Year']).copy()

Trim to the top 5,000 entries by global sales rank, as the remaining titles have very low sales.

In [ ]:
df = df[:5000].copy()
df

Rename the genre `"Platform"` to `"Platformer"` to avoid confusion with the `Platform` column.

In [ ]:
df['Genre'] = df['Genre'].map(lambda x: "Platformer" if x == "Platform" else x).copy()

Now we graph global sales over time, grouped by genre. Clicking a genre in the legend filters the bar chart to show total sales by publisher for that genre.

In [ ]:
sel = alt.selection_single(fields=['Genre'], bind="legend")

c1 = alt.Chart(df).mark_circle().encode(
    x='Year',
    y='Global_Sales',
    color='Genre:N',
    tooltip=['Name', 'Platform', 'Publisher']
).add_selection(sel)

c2 = alt.Chart(df).mark_bar().encode(
    x=alt.X('Publisher', sort='-y'),
    y='sum(Global_Sales)'
).transform_filter(sel)

c1 | c2

## Japan's Preferences (Regression)

We use scikit-learn's `LinearRegression` to identify what factors drive video game sales in Japan. We consider release year, the top 5 publishers, and genre. Since these are categorical variables, we encode them numerically using `OneHotEncoder`.

We filter to only the top 5 publishers to keep the model manageable.

In [ ]:
top_publishers = ['Nintendo', 'Electronic Arts', 'Activision', 'Sony Computer Entertainment', 'Ubisoft']
df2 = df[df['Publisher'].isin(top_publishers)].copy()

Convert the `Year` datetime column to an integer for use in the regression.

In [ ]:
df2['Year2'] = df2['Year'].dt.year

One-hot encode the `Publisher` column.

In [ ]:
encoder_pub = OneHotEncoder()
encoder_pub.fit(df2[["Publisher"]])

arr_pub = encoder_pub.transform(df2[["Publisher"]]).toarray()
pub_list = [encoder_pub.get_feature_names_out()[i][10:] for i in range(5)]
print(pub_list)

df2[pub_list] = arr_pub
df2

Repeat for the `Genre` column.

In [ ]:
encoder_g = OneHotEncoder()
encoder_g.fit(df2[["Genre"]])

arr_g = encoder_g.transform(df2[["Genre"]]).toarray()
g_list = [encoder_g.get_feature_names_out()[i][6:] for i in range(12)]
print(g_list)

df2[g_list] = arr_g
df2

Collect all encoded feature columns for the regression.

In [ ]:
cols = list(df2.columns)[10:]
cols

Fit the linear regression model predicting Japanese sales (`JP_Sales`) and inspect the coefficients.

In [ ]:
reg = LinearRegression(fit_intercept=False)
reg.fit(df2[cols], df2['JP_Sales'])

df2['Pred'] = reg.predict(df2[cols])

pd.Series(reg.coef_, index=cols)

Key observations from the coefficients:
- The `Year2` coefficient is negative, indicating Japanese sales declined over time.
- Nintendo has the highest publisher coefficient, reflecting its dominance in Japan.
- Role-Playing games have the highest genre coefficient, likely driven by franchises like Pokémon.

We visualize the regression prediction overlaid on the raw data, with publisher filtering via legend click.

In [ ]:
sel2 = alt.selection_single(fields=["Publisher"], bind="legend", empty="none")

c3 = alt.Chart(df2).mark_circle().encode(
    x='Year',
    y='JP_Sales',
    color='Publisher',
    tooltip=["Name", "Publisher", "Genre"],
    opacity=alt.condition(sel2, alt.value(1), alt.value(0.2)),
    size=alt.condition(sel2, alt.value(60), alt.value(20)),
).add_selection(sel2)

c4 = alt.Chart(df2).mark_line(color='black').encode(
    x='Year',
    y='Pred'
).transform_filter(sel2)

c3 + c4

Nintendo dominates the Japanese market, while American publishers like Electronic Arts and Activision have much lower Japanese sales.

## Platforms Used by Publishers (Classification)

We use a K-Nearest Neighbors classifier to predict the platform a game was released on, given publisher, sales figures, genre, and release year.

In [ ]:
feature_cols = [
    'Activision', 'Electronic Arts', 'Nintendo', 'Sony Computer Entertainment', 'Ubisoft',
    'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales',
    'Action', 'Adventure', 'Fighting', 'Misc', 'Platformer', 'Puzzle',
    'Racing', 'Role-Playing', 'Shooter', 'Simulation', 'Sports', 'Strategy',
    'Year2'
]

X = df2[feature_cols]
y = df2["Platform"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=2)

In [ ]:
clf = KNeighborsClassifier(n_neighbors=30)
clf.fit(X_train, y_train)

print(f"Training accuracy:  {clf.score(X_train, y_train):.4f}")
print(f"Test accuracy:      {clf.score(X_test, y_test):.4f}")

The training (~48.8%) and test (~44.6%) scores are close, indicating the model is not overfitting. However, the overall accuracy is low, suggesting that sales figures, publisher, and genre alone are insufficient to predict a game's platform.

## Summary

We analyzed video game sales trends across publishers, genres, and regions using a dataset of ~16,000 titles. Key steps included:

- **Data cleaning:** Removed nulls, unknown publishers, and the redundant `Rank` column. Converted `Year` to datetime.
- **Regression:** Used `LinearRegression` with one-hot encoded publisher and genre features to model Japanese sales. Found that Nintendo games and role-playing titles perform best in Japan, and that popularity has declined over time.
- **Classification:** Used `KNeighborsClassifier` (k=30) to predict release platform. Achieved ~45% test accuracy, indicating platform is difficult to predict from these features alone.

## References

- Dataset: [Video Game Sales — Kaggle (GregorySmith)](https://www.kaggle.com/datasets/gregorut/videogamesales)
- Sorted bar chart adapted from: https://altair-viz.github.io/gallery/bar_chart_sorted.html
- Lambda if/else syntax: https://thispointer.com/python-how-to-use-if-else-elif-in-lambda-functions/